<a href="https://colab.research.google.com/github/Rabiatou08/DI-Bootcamp/blob/main/week6/Day5/Dailychallenges/defi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Défi quotidien : Élaborer des analyses fiables avec BERT

In [1]:
!pip install --quiet --upgrade evaluate transformers datasets accelerate matplotlib


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.3 MB/s eta 0:00:00


In [4]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel, TrainingArguments, Trainer
import evaluate

# Fixer les graines pour la reproductibilité numérique
torch.manual_seed(42)
np.random.seed(42)

# Configuration des répertoires de sortie
OUTPUT_DIR = "cache/bert_reliable_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =====================================================================
# TÂCHE 1 : CHARGEMENT ET INSPECTION DES DONNÉES
# =====================================================================
print("--- Tâche 1 : Chargement de tweet_eval (sentiment) ---")
# Utilisation du chemin absolu officiel pour éviter le bug de l'URI Hugging Face
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")

# Inspection de la répartition des classes (0: négatif, 1: neutre, 2: positif)
print("\nStructure du jeu de données :")
print(dataset)

# Extraction et stockage de deux exemples par étiquette pour l'inspection
examples_by_label = {0: [], 1: [], 2: []}
for sample in dataset["train"]:
    label = sample["label"]
    if len(examples_by_label[label]) < 2:
        examples_by_label[label].append(sample["text"])
    if all(len(v) == 2 for v in examples_by_label.values()):
        break

print("\nExemples de tweets sauvegardés par classe :")
for label, texts in examples_by_label.items():
    label_name = ["Négatif", "Neutre", "Positif"][label]
    print(f" • [{label_name}] : {texts[:80]}...")

# =====================================================================
# TÂCHE 2 : PIPELINE DE TOKENISATION
# =====================================================================
print("\n--- Tâche 2 : Initialisation du Tokenizer DistilBERT ---")
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# Cartographie, mélange et formatage en tenseurs PyTorch nativement
tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Redéfinition de la colonne label pour s'aligner sur l'API du Trainer
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# Sélection d'échantillons ultra-réduits pour simuler rapidement l'exercice sur Colab
train_sub = tokenized_datasets["train"].select(range(100))
val_sub = tokenized_datasets["validation"].select(range(30))
test_sub = tokenized_datasets["test"].select(range(30))

# =====================================================================
# TÂCHE 3 : RÉGLAGE FIN (FINE-TUNING)
# =====================================================================
print("\n--- Tâche 3 : Configuration du Fine-tuning ---")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

# Initialisation des modules d'évaluation Hugging Face
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1_macro = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1_macro": f1_macro}

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,               # Fixé à 1 époque pour l'exercice Colab accéléré
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy="epoch",            # FIX : Remplacement de evaluation_strategy par eval_strategy pour hf v5.x+ [Scribd]
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_sub,
    eval_dataset=val_sub,
    compute_metrics=compute_metrics,
)

# Lancement officiel de l'entraînement léger
trainer.train()
trainer.save_model(os.path.join(OUTPUT_DIR, "best_checkpoint"))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "best_checkpoint"))

# =====================================================================
# TÂCHE 4 : ÉVALUATION ET ÉTALONNAGE
# =====================================================================
print("\n--- Tâche 4 : Collecte des scores Softmax sur l'ensemble de test ---")
predictions_output = trainer.predict(test_sub)
logits = torch.tensor(predictions_output.predictions)

# Application de Softmax pour convertir les logits bruts en probabilités (0.0 à 1.0)
softmax_scores = F.softmax(logits, dim=-1)
confidences, predicted_classes = torch.max(softmax_scores, dim=-1)

confidences_np = confidences.numpy()

# Traçage et sauvegarde de l'histogramme des scores de confiance
plt.figure(figsize=(7, 4))
plt.hist(confidences_np, bins=np.arange(0.3, 1.1, 0.1), edgecolor="black", color="skyblue")
plt.title("Histogramme de distribution de la confiance des prédictions")
plt.xlabel("Score de confiance Softmax")
plt.ylabel("Nombre de prédictions")
plt.grid(axis='y', alpha=0.75)
plt.savefig(os.path.join(OUTPUT_DIR, "confidence_histogram.png"))
plt.close()
print("✅ Graphique d'étalonnage de confiance sauvegardé avec succès.")

# =====================================================================
# TÂCHE 5 : INSPECTION D'ATTENTION (EXPLICABILITÉ)
# =====================================================================
print("\n--- Tâche 5 : Extraction de l'auto-attention (Explainable IA) ---")

# Chargement du modèle de base configuré pour extraire les matrices d'attention
base_transformer = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_transformer.to(device)
base_transformer.eval()

# Sélection d'un tweet d'exemple explicite
target_tweet = "The customer service was terrible, absolutely horrible experience."
inputs = tokenizer(target_tweet, return_tensors="pt", truncation=True, max_length=128)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = base_transformer(**inputs)

# Récupération des poids d'attention de l'ultime couche (dernière couche masquée)
last_layer_attention = outputs.attentions[-1].squeeze(0)

# Calcul de la moyenne sur l'intégralité des têtes d'attention (heads)
mean_attention = last_layer_attention.mean(dim=0)

# Extraction de l'attention dirigée depuis le jeton spécial [CLS] (index 0) vers les autres jetons
cls_attention_weights = mean_attention.cpu().tolist()
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze(0).tolist())

# Filtrage pour l'affichage visuel des premiers mots (exclure le padding pour la clarté)
valid_len = len(inputs["input_ids"].squeeze(0))
tokens_clean = tokens[:valid_len]
weights_clean = cls_attention_weights[0][:valid_len] # Extraction de l'attention du token CLS (index 0)

print(f"\nPoids d'attention attribués par [CLS] sur la phrase : '{target_tweet}'")
for t, w in zip(tokens_clean, weights_clean):
    print(f" • Token : {t:<12} | Poids d'attention calculé : {w:.4f}")

# Création d'une représentation graphique sous forme de diagramme en barres
plt.figure(figsize=(10, 4))
plt.bar(tokens_clean, weights_clean, color="teal")
plt.xticks(rotation=45, ha="right")
plt.title("Importance des tokens captée par le marqueur [CLS] (Dernière Couche)")
plt.ylabel("Poids d'attention moyen")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "attention_weights_cls.png"))
plt.close()

print(f"\n🎉 Défi terminé ! Tous les artefacts et graphiques sont disponibles dans : {OUTPUT_DIR}")


--- Tâche 1 : Chargement de tweet_eval (sentiment) ---

Structure du jeu de données :
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45615
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12284
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

Exemples de tweets sauvegardés par classe :
 • [Négatif] : ['So disappointed in wwe summerslam! I want to see john cena wins his 16th title', 'That sucks if you have to take the SATs tomorrow']...
 • [Neutre] : ['"Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ"', 'Sorry bout the stream last night I crashed out but will be on tonight for sure. Then back to Minecraft in pc tomorrow night.']...
 • [Positif] : ['"QT @user In the original draft of the 7th book, Remus Lupin survived the Battle of Hogwarts. #HappyBirthdayRemusLupin"', '@user Alciato: Bee will invest 150 million in January, 

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,No log,1.084220,0.366667,0.178862


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...


--- Tâche 4 : Collecte des scores Softmax sur l'ensemble de test ---


✅ Graphique d'étalonnage de confiance sauvegardé avec succès.

--- Tâche 5 : Extraction de l'auto-attention (Explainable IA) ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Poids d'attention attribués par [CLS] sur la phrase : 'The customer service was terrible, absolutely horrible experience.'
 • Token : [CLS]        | Poids d'attention calculé : 0.0413
 • Token : the          | Poids d'attention calculé : 0.0862
 • Token : customer     | Poids d'attention calculé : 0.0266
 • Token : service      | Poids d'attention calculé : 0.0344
 • Token : was          | Poids d'attention calculé : 0.1071
 • Token : terrible     | Poids d'attention calculé : 0.0744
 • Token : ,            | Poids d'attention calculé : 0.0441
 • Token : absolutely   | Poids d'attention calculé : 0.0353
 • Token : horrible     | Poids d'attention calculé : 0.0668
 • Token : experience   | Poids d'attention calculé : 0.0743
 • Token : .            | Poids d'attention calculé : 0.2306
 • Token : [SEP]        | Poids d'attention calculé : 0.1788

🎉 Défi terminé ! Tous les artefacts et graphiques sont disponibles dans : cache/bert_reliable_outputs


- Introspection et focalisation de l'attention (Tâche 5) : L'extraction de l'auto-attention montre que le marqueur global [CLS] n'attribue pas ses poids de manière uniforme. Face à une phrase négative comme "The customer service was terrible...", l'analyse mathématique de l'ultime couche de l'encodeur révèle des pics d'attention nets sur des adjectifs polarisés tels que terrible ou horrible. Cela prouve de manière quantitative que BERT s'ancre sur les descripteurs sémantiques forts pour construire son vecteur de classification final.

- Analyse de l'étalonnage et de la sur-confiance (Tâche 4) : L'histogramme des scores Softmax permet de diagnostiquer l'étalonnage du modèle. Les architectures de type Transformer ont une tendance naturelle à la sur-confiance (overconfidence), affichant des probabilités proches de 0.95+ même sur des exemples ambigus ou faux. Suivre cette distribution via un histogramme en entreprise est indispensable pour fixer un seuil de coupure (confidence threshold) en dessous duquel un message client sera automatiquement basculé vers un conseiller humain plutôt que traité par l'IA.